# Create IES Awards from official IES API

Creates Institute of Education Sciences (IES) awards from the official IES awards search API exposed by https://ies.ed.gov/use-work/awards.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/ies_to_s3.py` to download, write parquet, and upload the data.
- Contractor does not have S3/Databricks access; this notebook is prepared for admin handoff.

**Data source:** https://ies.ed.gov/use-work/awards  
**API endpoint:** https://api.ies.ed.gov/web/search/v1/execute  
**S3 location:** `s3a://openalex-ingest/awards/ies/ies_awards.parquet`

**Source decision:**
- The tracker explicitly says to CHECK FIRST for the IES grant search portal.
- The current IES awards page exposes an official search API that returned 3,183 Grant/Contract/Cooperative agreement records on 2026-05-16.
- The official API includes native award ID, title, description, awardee, amount, program, award status, office/center, topic tags, award date, content type, and IES landing-page path. This is the preferred source over USAspending for the OpenAlex IES funder.

**IES funder:**
- funder_id: 4320332210
- ROR: https://ror.org/04et59085
- DOI: 10.13039/100005246
- display_name: "Institute of Education Sciences"

**Expected raw fields:**
- `mid`: native IES award ID
- `mtitle`: award title
- `mdescriptionshort`: award description/summary
- `lawardamount`: award amount in USD
- `mdateprimary`: award date/year timestamp
- `mcontenttype`: Grant, Contract, or Cooperative agreement
- `lprogram`: IES/federal funding program
- `lawardee`: JSON array of awardee names
- `morgleaf`: JSON array of IES center/office labels
- `murl` / `landing_page_url`: IES award page


## Step 1: Create Staging Table from S3

In [ ]:
%sql
-- Create the staging table from S3 parquet
CREATE OR REPLACE TABLE openalex.awards.ies_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ies/ies_awards.parquet`;

In [ ]:
%sql
-- Check row count. Official IES API smoke test on 2026-05-16 returned 3,183 rows.
SELECT COUNT(*) as total_awards FROM openalex.awards.ies_awards_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT
    mid,
    mtitle,
    lawardamount,
    mdateprimary,
    mcontenttype,
    lprogram,
    lawardee,
    landing_page_url
FROM openalex.awards.ies_awards_raw
LIMIT 5;

## Step 1.5: Money-field scan + expected coverage

The official IES API exposes `lawardamount`, and the 2026-05-16 smoke test showed amount coverage across Grant/Contract/Cooperative agreement records. This scan is still mandatory per the runbook so that Databricks confirms the raw parquet has the expected money field before mapping. Currency is implicit USD because IES is a U.S. federal source.

In [ ]:
%sql
-- Money-field scan per runbook section 1.5
SELECT column_name FROM (DESCRIBE openalex.awards.ies_awards_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|obligated|outlay|grant|awarded';

## Step 1.6: Fail-fast - verify the IES funder row exists

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE funder_id = 4320332210`. If that row is not present, the CROSS JOIN silently emits zero rows and the insert can look successful. Assert presence here before transforming.

In [ ]:
%sql
-- Must return exactly 1 row with display_name = 'Institute of Education Sciences'.
-- If 0 rows, STOP and flag: the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320332210;

## Step 2: Create IES Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ies_awards
USING delta
AS
WITH
ies_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320332210  -- Institute of Education Sciences
),

awards_transformed AS (
    SELECT
        -- Generate unique ID using xxhash64 of funder_id:mid
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.mid))))) % 9000000000 as id,

        -- Display name = IES award title
        NULLIF(TRIM(g.mtitle), '') as display_name,

        -- Description = IES short description, falling back to title
        COALESCE(NULLIF(TRIM(g.mdescriptionshort), ''), NULLIF(TRIM(g.mtitle), '')) as description,

        -- Funder info
        f.funder_id,
        TRIM(g.mid) as funder_award_id,

        -- Amount (official IES awards are USD)
        TRY_CAST(g.lawardamount AS DOUBLE) as amount,
        'USD' as currency,

        -- Funder struct
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- Funding type from IES result type
        CASE
            WHEN LOWER(TRIM(g.mcontenttype)) = 'contract' THEN 'contract'
            WHEN LOWER(TRIM(g.mcontenttype)) = 'cooperative agreement' THEN 'cooperative_agreement'
            WHEN LOWER(TRIM(g.mcontenttype)) = 'grant' THEN 'grant'
            ELSE 'grant'
        END as funding_type,

        -- Funder scheme = IES program name
        NULLIF(TRIM(g.lprogram), '') as funder_scheme,

        -- Provenance: official IES API, not USAspending fallback
        'ies_official' as provenance,

        -- IES API exposes an award date/year timestamp, but not a separate end date
        TRY_TO_DATE(SUBSTR(g.mdateprimary, 1, 10), 'yyyy-MM-dd') as start_date,
        CAST(NULL AS DATE) as end_date,
        YEAR(TRY_TO_DATE(SUBSTR(g.mdateprimary, 1, 10), 'yyyy-MM-dd')) as start_year,
        CAST(NULL AS INT) as end_year,

        -- IES search results expose awardees but not PI names/ORCIDs in the bulk API
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        -- Landing page URL from official IES site
        CASE
            WHEN g.landing_page_url RLIKE '^https?://' THEN g.landing_page_url
            WHEN g.murl IS NOT NULL AND TRIM(g.murl) != '' THEN CONCAT('https://ies.ed.gov', g.murl)
            ELSE NULL
        END as landing_page_url,

        -- IES awards do not expose award DOIs
        CAST(NULL AS STRING) as doi,

        -- Works API URL
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.mid))))) % 9000000000) as works_api_url,

        -- Timestamps
        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.ies_awards_raw g
    CROSS JOIN ies_funder f
    WHERE g.mid IS NOT NULL
      AND TRIM(g.mid) != ''
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ies_official' AND priority = 61;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    61 as priority  -- IES priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.ies_awards;

## Verification Queries

In [ ]:
%sql
-- Count transformed awards
SELECT COUNT(*) as total_ies_awards FROM openalex.awards.ies_awards;

In [ ]:
%sql
-- Sample transformed awards
SELECT
    id,
    display_name,
    funder_award_id,
    amount,
    currency,
    funding_type,
    funder_scheme,
    start_year,
    landing_page_url
FROM openalex.awards.ies_awards
ORDER BY start_year DESC, amount DESC
LIMIT 20;

In [ ]:
%sql
-- Check funder distribution (should all be IES)
SELECT funder_id, funder.display_name, COUNT(*) as cnt
FROM openalex.awards.ies_awards
GROUP BY funder_id, funder.display_name;

In [ ]:
%sql
-- Check duplicate OpenAlex award IDs (should be zero rows)
SELECT id, COUNT(*) as cnt
FROM openalex.awards.ies_awards
GROUP BY id
HAVING COUNT(*) > 1;

In [ ]:
%sql
-- Check amount/currency coverage per runbook section 6.7
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN amount IS NOT NULL THEN 1 ELSE 0 END) AS has_amount,
    ROUND(SUM(CASE WHEN amount IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_with_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    MIN(currency) AS min_currency,
    MAX(currency) AS max_currency,
    ROUND(SUM(amount), 0) AS total_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount
FROM openalex.awards.ies_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook §6.3 form)
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(funder_award_id) as has_funder_award_id,
    COUNT(landing_page_url) as has_landing_page_url,
    COUNT(funder_scheme) as has_funder_scheme,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_dates
FROM openalex.awards.ies_awards;

In [ ]:
%sql
-- Funding type distribution
SELECT funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.ies_awards
GROUP BY funding_type
ORDER BY cnt DESC;

In [ ]:
%sql
-- Program distribution
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.ies_awards
GROUP BY funder_scheme
ORDER BY cnt DESC
LIMIT 30;

In [ ]:
%sql
-- Year distribution
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.ies_awards
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- Confirm insertion into openalex_awards_raw at priority 61
SELECT provenance, priority, COUNT(*) as cnt
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ies_official' AND priority = 61
GROUP BY provenance, priority;

In [ ]:
%sql
-- Raw source type sanity check
SELECT mcontenttype, COUNT(*) as cnt, SUM(TRY_CAST(lawardamount AS DOUBLE)) as total_amount
FROM openalex.awards.ies_awards_raw
GROUP BY mcontenttype
ORDER BY cnt DESC;